# Aula 10 — Laboratório: CNN × Tiny Vision Transformer

## CIFAR-10 — Gatos × Cachorros

Neste laboratório vamos comparar duas arquiteturas de redes neurais para classificar imagens do **CIFAR-10**:

- **CNN (Convolutional Neural Network)** — especializada em padrões espaciais locais;
- **Tiny Vision Transformer (TinyViT)** — trata a imagem como uma sequência de *patches* e usa **self-attention**.

### Pergunta experimental

> Em um experimento pequeno, treinado do zero, qual modelo apresenta melhor desempenho para diferenciar **gatos** e **cachorros**?

### O que vamos comparar

1. **Acurácia no conjunto de teste**
2. **Tempo de treinamento**
3. **Número de parâmetros treináveis**

> **Importante:** este experimento não busca determinar um “vencedor universal”.  
> Ele compara o comportamento dos dois modelos sob as mesmas condições de treinamento.

## 1. Bibliotecas

Se estiver usando um ambiente novo, instale antes:

```bash
pip install torch torchvision matplotlib pandas
```

O notebook usa apenas **PyTorch**, **TorchVision**, **Matplotlib**, **Pandas** e bibliotecas padrão do Python.

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

## 2. Reprodutibilidade e dispositivo

Usamos uma semente fixa para reduzir variações aleatórias entre execuções.

O PyTorch usará:

- **GPU CUDA**, se disponível;
- caso contrário, **CPU**.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo selecionado:", device)

## 3. Configuração do experimento

Para que a atividade seja rápida em sala de aula, usaremos somente parte das imagens de **gatos** e **cachorros**.

Você pode aumentar `N_TREINO`, `N_TESTE` e `EPOCHS` depois para observar como os resultados mudam.

In [ ]:
N_TREINO = 2000
N_TESTE = 500

BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001

# CIFAR-10:
# classe 3 = cat
# classe 5 = dog
CLASSES_ORIGINAIS = [3, 5]

NOMES_CLASSES = {
    0: "Gato",
    1: "Cachorro"
}

print("Treino:", N_TREINO)
print("Teste:", N_TESTE)
print("Batch size:", BATCH_SIZE)
print("Épocas:", EPOCHS)

## 4. Carregar o CIFAR-10

Cada imagem do CIFAR-10 possui:

\[
32 \times 32 \times 3
\]

ou seja:

- 32 pixels de largura;
- 32 pixels de altura;
- 3 canais de cor: **RGB**.

Os pixels são convertidos em tensores e normalizados aproximadamente para o intervalo `[-1, 1]`.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("CIFAR-10 treino completo:", len(train_dataset))
print("CIFAR-10 teste completo:", len(test_dataset))

## 5. Filtrar somente gatos e cachorros

No CIFAR-10 original:

- `3` = **cat**
- `5` = **dog**

Para o `CrossEntropyLoss`, vamos converter:

- gato → `0`
- cachorro → `1`

O filtro abaixo usa os rótulos internos do dataset (`dataset.targets`), evitando carregar imagens desnecessariamente apenas para descobrir suas classes.

In [ ]:
def selecionar_indices(dataset, classes, limite, seed=42):
    indices = [
        i for i, target in enumerate(dataset.targets)
        if target in classes
    ]

    rng = random.Random(seed)
    rng.shuffle(indices)

    if limite is not None:
        indices = indices[:limite]

    return indices


train_indices = selecionar_indices(
    train_dataset,
    CLASSES_ORIGINAIS,
    N_TREINO,
    seed=SEED
)

test_indices = selecionar_indices(
    test_dataset,
    CLASSES_ORIGINAIS,
    N_TESTE,
    seed=SEED
)

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Imagens de treino selecionadas:", len(train_subset))
print("Imagens de teste selecionadas:", len(test_subset))

In [ ]:
def converter_label(labels):
    # CIFAR-10:
    # 3 (cat) -> 0
    # 5 (dog) -> 1
    return (labels == 5).long()

## 6. Visualizar algumas imagens

A normalização precisa ser desfeita apenas para a visualização.

In [ ]:
def desnormalizar(img):
    return img * 0.5 + 0.5


images, labels = next(iter(train_loader))
labels_binarias = converter_label(labels)

n = min(8, len(images))

plt.figure(figsize=(12, 4))

for i in range(n):
    plt.subplot(2, 4, i + 1)

    img = desnormalizar(images[i]).permute(1, 2, 0).numpy()

    plt.imshow(np.clip(img, 0, 1))
    plt.title(NOMES_CLASSES[labels_binarias[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

# Modelo 1 — CNN

A CNN procura padrões locais na imagem usando filtros convolucionais.

Fluxo simplificado:

**Imagem → Convolução → ReLU → Pooling → Convolução → ReLU → Pooling → Classificador**

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                64 * 8 * 8,
                128
            ),
            nn.ReLU(),

            nn.Linear(
                128,
                2
            )
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
cnn = CNN().to(device)

print(cnn)

# Modelo 2 — Tiny Vision Transformer

O Vision Transformer divide a imagem em pequenos **patches**.

Neste exemplo:

- imagem: `32 × 32`;
- patch: `4 × 4`;
- teremos `8 × 8 = 64` patches;
- cada patch é convertido em um vetor;
- os vetores são processados pelo **Transformer Encoder**.

Fluxo simplificado:

**Imagem → Patches → Embeddings → Self-Attention → Transformer → Classificador**

In [ ]:
class TinyViT(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=4,
        dim=64,
        heads=4,
        layers=2
    ):
        super().__init__()

        self.patch_size = patch_size

        # Patch embedding:
        # uma convolução com kernel=stride=patch_size
        # divide a imagem em regiões sem sobreposição.
        self.patch_embedding = nn.Conv2d(
            in_channels=3,
            out_channels=dim,
            kernel_size=patch_size,
            stride=patch_size
        )

        num_patches = (image_size // patch_size) ** 2

        # Informação aprendível sobre a posição dos patches.
        self.position_embedding = nn.Parameter(
            torch.randn(1, num_patches, dim) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

        self.classifier = nn.Linear(
            dim,
            2
        )

    def forward(self, x):

        # Entrada:
        # B x 3 x 32 x 32

        x = self.patch_embedding(x)

        # Após patch embedding:
        # B x dim x 8 x 8

        x = x.flatten(2)

        # B x dim x 64

        x = x.transpose(1, 2)

        # B x 64 x dim

        x = x + self.position_embedding

        x = self.transformer(x)

        # Global average pooling entre os tokens visuais.
        x = x.mean(dim=1)

        x = self.classifier(x)

        return x

In [ ]:
vit = TinyViT().to(device)

print(vit)

## 7. Quantos parâmetros cada modelo possui?

Essa é uma das métricas propostas no slide 29.

Um **parâmetro treinável** é um valor que pode ser ajustado pelo algoritmo de treinamento.

In [ ]:
def contar_parametros(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


parametros_cnn = contar_parametros(cnn)
parametros_vit = contar_parametros(vit)

print(f"CNN:     {parametros_cnn:,} parâmetros")
print(f"TinyViT: {parametros_vit:,} parâmetros")

# 8. Função de treinamento

O ciclo é o mesmo para os dois modelos:

1. carregar um **batch**;
2. executar o **forward**;
3. calcular o **erro (loss)**;
4. executar **backpropagation**;
5. atualizar os pesos com o **optimizer**;
6. seguir para o próximo batch.

A arquitetura muda, mas o princípio de aprendizagem permanece o mesmo.

In [ ]:
def treinar(
    model,
    train_loader,
    epochs=3,
    learning_rate=0.001
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    historico = {
        "loss": [],
        "accuracy": []
    }

    inicio = time.perf_counter()

    for epoch in range(epochs):

        model.train()

        total_loss = 0.0
        corretos = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = converter_label(labels).to(device)

            # 1. Zera gradientes anteriores
            optimizer.zero_grad()

            # 2. Forward
            outputs = model(images)

            # 3. Loss
            loss = criterion(outputs, labels)

            # 4. Backpropagation
            loss.backward()

            # 5. Atualização dos pesos
            optimizer.step()

            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            corretos += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

        loss_medio = total_loss / len(train_loader)
        accuracy = 100 * corretos / total

        historico["loss"].append(loss_medio)
        historico["accuracy"].append(accuracy)

        print(
            f"Época {epoch + 1:02d}/{epochs} | "
            f"Loss: {loss_medio:.4f} | "
            f"Acurácia treino: {accuracy:.2f}%"
        )

    tempo = time.perf_counter() - inicio

    return tempo, historico

# 9. Função de avaliação

Durante a avaliação:

- não calculamos gradientes;
- não atualizamos os pesos;
- apenas medimos a capacidade do modelo em dados que não foram usados para o treinamento.

In [ ]:
def avaliar(model, test_loader):

    model.eval()

    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    corretos = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = converter_label(labels).to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            corretos += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    loss_medio = total_loss / len(test_loader)
    accuracy = 100 * corretos / total

    return loss_medio, accuracy

# 10. Treinar a CNN

In [ ]:
# Reinicializar o modelo antes do experimento
cnn = CNN().to(device)

tempo_cnn, historico_cnn = treinar(
    cnn,
    train_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

loss_teste_cnn, acc_cnn = avaliar(
    cnn,
    test_loader
)

parametros_cnn = contar_parametros(cnn)

print()
print("CNN")
print(f"Loss teste:       {loss_teste_cnn:.4f}")
print(f"Acurácia teste:   {acc_cnn:.2f}%")
print(f"Tempo treino:     {tempo_cnn:.2f} s")
print(f"Parâmetros:       {parametros_cnn:,}")

# 11. Treinar o Tiny Vision Transformer

In [ ]:
# Reinicializar o modelo antes do experimento
vit = TinyViT().to(device)

tempo_vit, historico_vit = treinar(
    vit,
    train_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

loss_teste_vit, acc_vit = avaliar(
    vit,
    test_loader
)

parametros_vit = contar_parametros(vit)

print()
print("Tiny Vision Transformer")
print(f"Loss teste:       {loss_teste_vit:.4f}")
print(f"Acurácia teste:   {acc_vit:.2f}%")
print(f"Tempo treino:     {tempo_vit:.2f} s")
print(f"Parâmetros:       {parametros_vit:,}")

# 12. Comparação final

Agora preenchemos a tabela proposta no slide 29:

| Modelo | Acurácia | Tempo | Parâmetros |
|---|---:|---:|---:|
| CNN | ? | ? | ? |
| TinyViT | ? | ? | ? |

Os valores abaixo são calculados automaticamente após o treinamento.

In [ ]:
resultado = pd.DataFrame({
    "Modelo": [
        "CNN",
        "Tiny Vision Transformer"
    ],
    "Acurácia teste (%)": [
        acc_cnn,
        acc_vit
    ],
    "Tempo treinamento (s)": [
        tempo_cnn,
        tempo_vit
    ],
    "Parâmetros treináveis": [
        parametros_cnn,
        parametros_vit
    ]
})

resultado

## 13. Curvas de aprendizagem

As curvas ajudam a observar se o modelo está realmente aprendendo durante as poucas épocas utilizadas no experimento.

In [ ]:
epocas = range(1, EPOCHS + 1)

plt.figure(figsize=(8, 5))

plt.plot(
    epocas,
    historico_cnn["accuracy"],
    marker="o",
    label="CNN"
)

plt.plot(
    epocas,
    historico_vit["accuracy"],
    marker="o",
    label="TinyViT"
)

plt.xlabel("Época")
plt.ylabel("Acurácia de treino (%)")
plt.title("CNN × TinyViT — evolução da acurácia")
plt.xticks(list(epocas))
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    epocas,
    historico_cnn["loss"],
    marker="o",
    label="CNN"
)

plt.plot(
    epocas,
    historico_vit["loss"],
    marker="o",
    label="TinyViT"
)

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("CNN × TinyViT — evolução da função de perda")
plt.xticks(list(epocas))
plt.legend()
plt.grid(alpha=0.3)

plt.show()

# 14. Visualizar algumas previsões

Vamos observar imagens do conjunto de teste e comparar:

- classe verdadeira;
- previsão da CNN;
- previsão do TinyViT.

In [ ]:
def prever(model, images):
    model.eval()

    with torch.no_grad():
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)

    return predictions.cpu()


images, labels = next(iter(test_loader))

labels_reais = converter_label(labels)

pred_cnn = prever(cnn, images)
pred_vit = prever(vit, images)

n = min(8, len(images))

plt.figure(figsize=(14, 7))

for i in range(n):

    plt.subplot(2, 4, i + 1)

    img = desnormalizar(images[i]).permute(1, 2, 0).numpy()

    real = NOMES_CLASSES[labels_reais[i].item()]
    cnn_pred = NOMES_CLASSES[pred_cnn[i].item()]
    vit_pred = NOMES_CLASSES[pred_vit[i].item()]

    plt.imshow(np.clip(img, 0, 1))

    plt.title(
        f"Real: {real}\n"
        f"CNN: {cnn_pred} | ViT: {vit_pred}",
        fontsize=9
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

# 15. Interpretação

### Se a CNN obtiver maior acurácia, o Transformer falhou?

**Não.**

Neste experimento:

- temos poucas imagens;
- treinamos por poucas épocas;
- o TinyViT começa **do zero**;
- não usamos pré-treinamento em grandes bases.

As CNNs possuem uma estrutura muito adequada a padrões locais de imagens.  
Vision Transformers costumam se beneficiar bastante de **mais dados** e de **pré-treinamento**.

### Perguntas para discussão

1. Qual modelo obteve maior acurácia?
2. Qual modelo treinou mais rapidamente?
3. Qual possui mais parâmetros?
4. A diferença entre os modelos aumenta se treinarmos por mais épocas?
5. O resultado muda aumentando o conjunto de treinamento?
6. O TinyViT melhora se aumentarmos o número de camadas ou cabeças de atenção?
7. Uma arquitetura mais moderna é necessariamente melhor em qualquer problema?

> **Conclusão:** arquitetura, quantidade de dados, capacidade computacional e estratégia de treinamento precisam ser analisadas em conjunto.

## 16. Experimentos opcionais

Depois de executar o notebook básico, altere **uma variável por vez**.

### Experimento A — mais épocas

```python
EPOCHS = 10
```

### Experimento B — mais dados

```python
N_TREINO = 8000
N_TESTE = 2000
```

> Depois de alterar `N_TREINO` ou `N_TESTE`, execute novamente as células que criam os subconjuntos e os `DataLoader`.

### Experimento C — patches maiores

No TinyViT:

```python
patch_size = 8
```

Isso reduz a quantidade de tokens visuais.

### Experimento D — mais camadas Transformer

```python
layers = 4
```

### Experimento E — mais cabeças de atenção

```python
heads = 8
```

Observe como acurácia, tempo e quantidade de parâmetros se alteram.